# Spectrum Tokenizer

- `n_top_peaks`: The maximum number of peaks to retain per spectrum, selected by highest intensity.
- `min_mz`: The minimum $m/z$ value. Peaks below this threshold are filtered out.
- `max_mz`: The maximum $m/z$ value. Peaks above this threshold are filtered out.
- `min_intensity`: The minimum intensity threshold. Peaks below this value are removed.
- `remove_precursor_tol`: The mass tolerance window used to remove precursor ion peaks from the spectrum.

In [1]:
import sys
sys.path.append('..')
import random
from dataclasses import dataclass

import h5py
import numpy.typing as npt

from rocnovo.tokenizer.spectrum import SpectrumTokenizer

@dataclass
class Spectrum:
    mz_array: npt.NDArray
    int_array: npt.NDArray
    precursor_mz: float
    precursor_charge: int

def get_spectrum(idx: int | None=None):
    # replace the path to your own peaks.db.hdf5
    with h5py.File("/data2/xp/RocNovo-Lightning/outputs/tutorials/data/processed/peaks.db.hdf5") as f:
        # you can view the rocnovo/data/datasets.py
        # we just copy the __getitem__ method
        stream_handle = f["0"]
        # random choose a spectrum
        if idx is None:
            random_index = random.randint(0, stream_handle.attrs["n_spectra"] - 1)
        else:
            random_index = idx
        
        print(f"choose the {random_index}th spectrum")
        start_offset = stream_handle["metadata"][random_index]["offset"]
        precursor_mz = stream_handle["metadata"][random_index]["precursor_mz"]
        precursor_charge = stream_handle["metadata"][random_index]["precursor_charge"]
        if random_index == stream_handle.attrs["n_spectra"] - 1:
            stop_offset = stream_handle.attrs["n_peaks"]
        else:
            stop_offset = stream_handle["metadata"][random_index + 1]["offset"]
        
        peaks = stream_handle["spectra"][start_offset:stop_offset]
        mz_array = peaks["mz_array"]
        int_array = peaks["intensity_array"]
        return Spectrum(
            mz_array,
            int_array,
            precursor_mz,
            precursor_charge
        )

# tokenize the spectrum
spectrum_tokenizer = SpectrumTokenizer(
    150,
    50.0,
    2500.0,
    0.01,
    2
)
spectrum = get_spectrum(26592)
tokenized_peaks = spectrum_tokenizer.tokenize(
    spectrum.mz_array,
    spectrum.int_array,
    spectrum.precursor_mz,
    spectrum.precursor_charge
)

choose the 26592th spectrum


We can see that the number of peaks in the tokenized spectrum is 51, which is less than the original number of peaks 93.

In [2]:
spectrum.mz_array.shape[0], tokenized_peaks.shape[0]

(93, 51)

# Spectrum Augmentation

There are two data augmentation methods in our project. 

-  Returning random noise data, which was used in the paper's experiments.
-  Realistic random peak masking and random intensity shifting operations.


Augmentation is only used during the CLIP training phase and is disabled on the validation set.

We adopted the first method because our experiments were conducted on two 4090 GPUs. Since the model was trained in FP32 precision, the global batch size was limited to 400, which caused the model to converge too quickly. We found that returning random noise data helps alleviate this overfitting.

If you use a larger batch size, we recommend using the second method. We also test the second method on two 4090 GPU with bf16 mixed precision.

In [3]:
from rocnovo.config.aug import AugmentationConfig

# we can use the enabled parameter to control whether to enable data augmentation.
# return_dummy_tensor parameter to control whether to return dummy tensor.
aug_config = AugmentationConfig(
    True,
    0.5,
    0.2,
    0.3,
    0.15,
    return_dummy_tensor=True
)
# you can run multiple times to get different results.
spectrum_tokenizer = SpectrumTokenizer(
    150,
    50.0,
    2500.0,
    0.01,
    2
)
spectrum_tokenizer.set_aug_config(aug_config)
spectrum = get_spectrum(6613)
tokenized_peaks = spectrum_tokenizer.tokenize(
    spectrum.mz_array,
    spectrum.int_array,
    spectrum.precursor_mz,
    spectrum.precursor_charge
)
tokenized_peaks.shape[0], tokenized_peaks

choose the 6613th spectrum


(1, tensor([[0., 1.]]))

In [4]:
from rocnovo.config.aug import AugmentationConfig

# we can use the enabled parameter to control whether to enable data augmentation.
# return_dummy_tensor parameter to control whether to return dummy tensor.
aug_config = AugmentationConfig(
    True,
    0.5,
    0.2,
    0.3,
    0.15,
    return_dummy_tensor=False
)
# you can run multiple times to get different results.
spectrum_tokenizer = SpectrumTokenizer(
    150,
    50.0,
    2500.0,
    0.01,
    2
)
spectrum_tokenizer.set_aug_config(aug_config)
spectrum = get_spectrum(20696)
tokenized_peaks = spectrum_tokenizer.tokenize(
    spectrum.mz_array,
    spectrum.int_array,
    spectrum.precursor_mz,
    spectrum.precursor_charge
)
tokenized_peaks.shape[0], tokenized_peaks

choose the 20696th spectrum


(58,
 tensor([[1.2907e+02, 1.0009e-01],
         [1.2910e+02, 2.4276e-01],
         [1.3009e+02, 2.0218e-01],
         [1.3306e+02, 3.7696e-01],
         [1.3608e+02, 1.8288e-01],
         [1.4711e+02, 2.8544e-01],
         [1.6706e+02, 1.0792e-01],
         [1.7512e+02, 2.0921e-01],
         [1.9911e+02, 1.1000e-01],
         [2.1309e+02, 2.0484e-01],
         [2.1610e+02, 1.2663e-01],
         [2.1912e+02, 1.5801e-01],
         [2.2608e+02, 1.3203e-01],
         [2.2707e+02, 2.7739e-01],
         [2.4215e+02, 1.4019e-01],
         [2.4409e+02, 7.1318e-01],
         [2.4618e+02, 2.8147e-01],
         [2.5514e+02, 1.1651e-01],
         [2.6608e+02, 2.0203e-01],
         [2.6809e+02, 1.0995e-01],
         [2.7617e+02, 1.0508e-01],
         [2.8211e+02, 1.1934e-01],
         [2.8610e+02, 1.7658e-01],
         [2.9906e+02, 2.4414e-01],
         [2.9913e+02, 1.6490e-01],
         [3.0106e+02, 1.6628e-01],
         [3.2115e+02, 1.4165e-01],
         [3.2713e+02, 1.6522e-01],
         [3.421

# Peptide Tokenizer

- `residues`: Defines the amino acid vocabulary and their exact masses.
  - `"canonical"`: Uses the 20 standard amino acids (with Cysteine carbamidomethylation by default).
  - `"massivekb"`: Expands the vocabulary to include common modifications found in the MassIVE-KB dataset.
  - Custom dictionary: You can also pass a custom dictionary mapping specific residues or modifications to their masses.
- `max_len`: The maximum allowed length for a peptide sequence. Sequences exceeding this length are truncated or filtered.
- `reverse`: If `True`, the peptide sequence is reversed. This is particularly useful for bidirectional decoding strategies.

## Using the MassiveKB

In [5]:
from rocnovo.tokenizer.peptide import PTMPeptideTokenizer

# we will replace "I" with "L"
peptide_tokenizer = PTMPeptideTokenizer(
    "massivekb",
    reverse=True
)

In [6]:
tokens = peptide_tokenizer.tokenize("PEPTIDE")
tokens

tensor([ 1, 15, 12,  9,  7,  5, 15,  5,  1])

In [7]:
peptide_tokenizer.detokenize_seqence(tokens)

'PEPTLDE'

In [8]:
peptide_tokenizer.idx2vocab, peptide_tokenizer.vocab2idx

({2: 'G',
  3: 'A',
  4: 'S',
  5: 'P',
  6: 'V',
  7: 'T',
  8: 'C+57.021',
  9: 'L',
  10: 'I',
  11: 'N',
  12: 'D',
  13: 'Q',
  14: 'K',
  15: 'E',
  16: 'M',
  17: 'H',
  18: 'F',
  19: 'R',
  20: 'Y',
  21: 'W',
  22: '+42.011',
  23: '+43.006',
  24: '-17.027',
  25: '+43.006-17.027',
  26: 'M+15.995',
  27: 'N+0.984',
  28: 'Q+0.984',
  0: 'PAD',
  1: 'SOS'},
 {'G': 2,
  'A': 3,
  'S': 4,
  'P': 5,
  'V': 6,
  'T': 7,
  'C+57.021': 8,
  'L': 9,
  'I': 10,
  'N': 11,
  'D': 12,
  'Q': 13,
  'K': 14,
  'E': 15,
  'M': 16,
  'H': 17,
  'F': 18,
  'R': 19,
  'Y': 20,
  'W': 21,
  '+42.011': 22,
  '+43.006': 23,
  '-17.027': 24,
  '+43.006-17.027': 25,
  'M+15.995': 26,
  'N+0.984': 27,
  'Q+0.984': 28,
  'PAD': 0,
  'SOS': 1})

## Custom Tokenizers

In [9]:
from rocnovo.tokenizer.peptide import PTMPeptideTokenizer

residues = {
    "G": 57.021463735,
    "A": 71.037113805,
    "S": 87.032028435,
    "P": 97.052763875,
    "V": 99.068413945,
    "T": 101.047678505,
    "C(+57.02)": 103.009184505 + 57.02146,
    "L": 113.084064015,
    "I": 113.084064015,
    "N": 114.042927470,
    "D": 115.026943065,
    "Q": 128.058577540,
    "K": 128.094963050,
    "E": 129.042593135,
    "M": 131.040484645,
    "H": 137.058911875,
    "F": 147.068413945,
    "R": 156.101111050,
    "Y": 163.063328575,
    "W": 186.079312980,
    "M(+15.99)": 131.040484645 + 15.994915,
}
peptide_tokenizer = PTMPeptideTokenizer(
    residues,
    reverse=True
)

In [10]:
peptide_tokenizer.vocab2idx, peptide_tokenizer.idx2vocab

({'G': 2,
  'A': 3,
  'S': 4,
  'P': 5,
  'V': 6,
  'T': 7,
  'C(+57.02)': 8,
  'L': 9,
  'I': 10,
  'N': 11,
  'D': 12,
  'Q': 13,
  'K': 14,
  'E': 15,
  'M': 16,
  'H': 17,
  'F': 18,
  'R': 19,
  'Y': 20,
  'W': 21,
  'M(+15.99)': 22,
  'PAD': 0,
  'SOS': 1},
 {2: 'G',
  3: 'A',
  4: 'S',
  5: 'P',
  6: 'V',
  7: 'T',
  8: 'C(+57.02)',
  9: 'L',
  10: 'I',
  11: 'N',
  12: 'D',
  13: 'Q',
  14: 'K',
  15: 'E',
  16: 'M',
  17: 'H',
  18: 'F',
  19: 'R',
  20: 'Y',
  21: 'W',
  22: 'M(+15.99)',
  0: 'PAD',
  1: 'SOS'})

In [11]:
tokens = peptide_tokenizer.tokenize("C(+57.02)M(+15.99)MRQ")
tokens

tensor([ 1, 13, 19, 16, 22,  8,  1])

In [12]:
peptide_tokenizer.detokenize(tokens)

['C(+57.02)', 'M(+15.99)', 'M', 'R', 'Q']